# SimSwap Setup and Synthesis (generative component)

**Cross-Generator Generalization in Deepfake Detection (IE7374)**

Gets you from nothing to a real SimSwap "unseen generator" test set, added to the
shared `crops.parquet` manifest. Run this on the same machine/VM where your FF++
`data/raw/real/*.mp4` already lives -- this does **not** need `data/preprocess.py`
to have run first (see the note in Step 0), only the raw download.

**Honesty check before you start:** `models/simswap_generator.py` has never been
run against a real cloned SimSwap repo in this project -- its exact API
(`create_model`, checkpoint filenames, the `_Opt` fields) was written by reading
SimSwap's own inference scripts, not verified against a live install. Budget time
for a first-run import/attribute mismatch; Step 4 below is specifically designed
to surface that early, before you've spent time generating anything.

## 0. Confirm prerequisites
Only needs the raw download (Phase 1), not the crops cache (Phase 3) --
`synthesize_simswap.py` does its own frame sampling and face cropping internally,
independent of `data/preprocess.py`.

In [ ]:
import os
print("repo root:", os.getcwd())
n_real = len([f for f in os.listdir("data/raw/real") if f.endswith(".mp4")]) if os.path.isdir("data/raw/real") else 0
print("real clips available:", n_real)
assert n_real >= 2, "need data/raw/real/*.mp4 -- run 00_setup_and_preprocess.ipynb steps 1-5 first"


## 1. Clone the SimSwap repo
Separate from your project repo -- `models/simswap_generator.py` imports SimSwap's
own model code from wherever you clone this, passed as `--simswap-repo`.

In [ ]:
SIMSWAP_REPO = "/content/SimSwap"   # ADJUST if not on Colab

if not os.path.isdir(SIMSWAP_REPO):
    !git clone https://github.com/neuralchen/SimSwap.git {SIMSWAP_REPO}
else:
    print("already cloned:", SIMSWAP_REPO)


## 2. Install SimSwap's dependencies
`insightface` handles face detection/alignment for SimSwap specifically (separate
from mediapipe, which the rest of this project's pipeline uses). Not currently in
`requirements.txt` -- add it there if this becomes a standing part of the pipeline.

In [ ]:
!pip install -q insightface onnxruntime-gpu

## 3. Download pretrained weights
Two of these are on stable GitHub Releases (scriptable, no manual step). The third
-- the `antelope` face-detection model `insightface.FaceAnalysis` needs -- is the
fragile one: SimSwap's own repo links it via a personal OneDrive URL, not a stable
release. `insightface` usually auto-downloads named model packs like `antelope`
the first time `FaceAnalysis(name=...)` runs, from insightface's own model zoo --
try that first (Step 4 will tell you immediately if it fails) before falling back
to a manual download.

In [ ]:
WEIGHTS_DIR = "simswap_weights"   # ADJUST if you want it elsewhere
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(f"{WEIGHTS_DIR}/arcface_model", exist_ok=True)

# arcface checkpoint -- stable GitHub release URL
!wget -q -nc -P {WEIGHTS_DIR}/arcface_model \
    https://github.com/neuralchen/SimSwap/releases/download/1.0/arcface_checkpoint.tar

# main SimSwap checkpoints (includes simswap_224.pth under checkpoints/people/)
!wget -q -nc -O {WEIGHTS_DIR}/checkpoints.zip \
    https://github.com/neuralchen/SimSwap/releases/download/1.0/checkpoints.zip
!unzip -q -n {WEIGHTS_DIR}/checkpoints.zip -d {WEIGHTS_DIR}/checkpoints

print("downloaded:")
!find {WEIGHTS_DIR} -maxdepth 3 -type f


**Note the layout this produces:** `models/simswap_generator.py`'s `_Opt` class
expects `Arc_path = os.path.join(weights_dir, "arcface_checkpoint.tar")` directly
under `weights_dir` (not the `arcface_model/` subfolder the zip creates) and
`checkpoints_dir = weights_dir` with a `people/` subfolder inside it (matching
`_Opt.name = "people"`). The cell below normalizes the layout to match what the
wrapper actually expects, rather than editing the wrapper.

In [ ]:
import shutil

# flatten arcface_checkpoint.tar to directly under WEIGHTS_DIR
src = f"{WEIGHTS_DIR}/arcface_model/arcface_checkpoint.tar"
dst = f"{WEIGHTS_DIR}/arcface_checkpoint.tar"
if os.path.exists(src) and not os.path.exists(dst):
    shutil.move(src, dst)

print("Arc_path exists:", os.path.exists(dst))
print("people checkpoint dir exists:", os.path.isdir(f"{WEIGHTS_DIR}/checkpoints/people"))
!ls {WEIGHTS_DIR}/checkpoints/people 2>/dev/null || echo "not found -- check the unzip step above"


## 4. Smoke-test `SimSwapGenerator` -- the real first-ever run
This is the step that actually validates (or breaks) the untested assumptions in
`models/simswap_generator.py`. If this cell fails, the traceback tells you exactly
which assumption was wrong (import path, `_Opt` field name, checkpoint layout) --
fix the wrapper based on the real error rather than guessing further.

In [ ]:
import sys
sys.path.append(os.getcwd())  # so "from models.simswap_generator import SimSwapGenerator" resolves

from models.simswap_generator import SimSwapGenerator

gen = SimSwapGenerator(weights_dir=WEIGHTS_DIR, simswap_repo=SIMSWAP_REPO)
print("SimSwapGenerator loaded successfully")


**If Step 4 failed on the `antelope` model specifically** (an `insightface`
download or file-not-found error), that's the fragile fallback case flagged in
Step 3. Manual fallback:
```python
!wget --no-check-certificate "https://sh23tw.dm.files.1drv.com/y4mmGiIkNVigkSwOKDcV3nwMJulRGhbtHdkheehR5TArc52UjudUYNXAEvKCii2O5LAmzGCGK6IfleocxuDeoKxDZkNzDRSt4ZUlEt8GlSOpCXAFEkBwaZimtWGDRbpIGpb_pz9Nq5jATBQpezBS6G_UtspWTkgrXHHxhviV2nWy8APPx134zOZrUIbkSF6xnsqzs3uZ_SEX_m9Rey0ykpx9w" -O antelope.zip
!unzip -q antelope.zip -d {WEIGHTS_DIR}/models/antelope
```
This is a personal OneDrive link from SimSwap's own repo -- it may be stale or
rate-limited by the time you read this. If it doesn't work, search SimSwap's
GitHub issues for "antelope" for a current mirror.

## 5. One manual swap, visually checked
Before generating a whole batch, swap two real clips' faces once and look at it --
cheaper than discovering a systematic problem after generating 200 pairs.

In [ ]:
import cv2
import matplotlib.pyplot as plt

real_clips = sorted(f for f in os.listdir("data/raw/real") if f.endswith(".mp4"))[:2]
source_path = f"data/raw/real/{real_clips[0]}"
target_path = f"data/raw/real/{real_clips[1]}"

cap = cv2.VideoCapture(source_path)
_, source_frame = cap.read()
cap.release()

cap = cv2.VideoCapture(target_path)
_, target_frame = cap.read()
cap.release()

swapped = gen.swap(source_frame, target_frame)
if swapped is None:
    print("swap returned None -- no face detected in source or target frame; try a different pair")
else:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, img, title in zip(axes,
                               [source_frame, target_frame, swapped],
                               ["source identity", "target frame", "swapped result"]):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 6. Generate the full SimSwap "unseen generator" set
`--manifest` points at your team's **real, existing** `crops.parquet` (not a fresh
one) so the SimSwap rows land in the same file training actually reads from.
`--pairs` controls how many source/target swaps to generate; start smaller to
confirm throughput before committing to the full run.

Skip cell 5's failure mode here too: expect some `n_fail` count from clips where no
face was detected in a sampled frame -- that's normal, not a bug, as long as
`n_ok` is the large majority.

In [ ]:
PAIRS = 50   # ADJUST -- start small, raise once you've confirmed this runs cleanly

!python data/synthesize_simswap.py \
    --raw data/raw/real \
    --weights-dir {WEIGHTS_DIR} \
    --simswap-repo {SIMSWAP_REPO} \
    --manifest data/manifests/crops.parquet \
    --split-out data/splits/simswap-test.csv \
    --pairs {PAIRS}


## 7. Verify the SimSwap rows landed in the shared manifest

In [ ]:
import pandas as pd

m = pd.read_parquet("data/manifests/crops.parquet")
simswap_rows = m[m["method"] == "SimSwap"]
print("total crops in manifest:", len(m))
print("SimSwap crops:", len(simswap_rows))
print(m.groupby("method").size())

split = pd.read_csv("data/splits/simswap-test.csv")
print("\nsimswap-test.csv rows:", len(split), " unique roles:", split["role"].unique().tolist())


In [ ]:
# eyeball one swapped crop
row = simswap_rows.iloc[0]
import numpy as np, matplotlib.pyplot as plt
img = np.load(row["path"])
plt.imshow(img); plt.axis("off")
plt.title(f"{row['crop_id']}  ({img.shape})")
plt.show()


## 8. Share with the team
Same pattern as your other results: the `.npy` crops are git-ignored, but the
manifest update and the new split file are small and should be committed so every
teammate's `evaluate.py` picks up the `SimSwap` column automatically (per
`extra_test_sets` in each run's config -- no per-run code change needed).

In [ ]:
# !git add data/splits/simswap-test.csv
# !git commit -m "Add SimSwap unseen-generator set"
# !git pull --rebase && git push
print("uncomment above once you've confirmed the results look right")


## Next steps
- If a teammate re-runs `experiments/evaluate.py` on an existing checkpoint, the
  `SimSwap` column now populates automatically -- no retraining needed for runs
  already done, only re-evaluation.
- Runs still to be trained will pick up the SimSwap column on their first
  `evaluate.py` pass, same as any other extra test set.
- Consider raising `--pairs` for a larger SimSwap set once this pipeline is
  confirmed stable, matching the FF++ methods' scale where feasible.